# EDA — Sentiment Analysis Dataset

Exploratory Data Analysis of the unified dataset collected by `DataCollectionAgent`.

**ML Task:** Binary sentiment classification — predict `positive` / `negative`.

**Sources:**
| Source | Domain | Labelling |
|--------|--------|-----------|
| `imdb_huggingface` | Movie reviews (full text) | Human annotation |
| `books_toscrape` | Book titles | Star rating (1-2★=neg, 4-5★=pos) |
| `openlibrary_api` | Book titles | Community avg rating |

**Schema:** `text`, `label`, `source`, `collected_at`

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

DATA_PATH = pathlib.Path('../data/raw/unified_dataset.csv')
assert DATA_PATH.exists(), f'Run  python run_agent.py  first! ({DATA_PATH} not found)'

df = pd.read_csv(DATA_PATH)
print(f'Rows: {len(df):,}   |   Columns: {list(df.columns)}')
df.head()

## 1. Dataset overview

In [ ]:
print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nUnique values per column:')
print(df.nunique())

## 2. Source & label distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 2a — records per source (pie)
src_counts = df['source'].value_counts()
axes[0].pie(src_counts.values, labels=src_counts.index, autopct='%1.1f%%',
            startangle=140, colors=sns.color_palette('muted', len(src_counts)))
axes[0].set_title('Records per source', fontweight='bold')

# 2b — overall label balance
lbl_counts = df['label'].value_counts()
colors = {'positive': '#4CAF50', 'negative': '#F44336'}
bar_colors = [colors.get(l, '#90A4AE') for l in lbl_counts.index]
axes[1].bar(lbl_counts.index, lbl_counts.values, color=bar_colors, edgecolor='white')
axes[1].set_title('Overall label balance', fontweight='bold')
axes[1].set_ylabel('Count')
for i, (lbl, cnt) in enumerate(lbl_counts.items()):
    axes[1].text(i, cnt + 5, str(cnt), ha='center', fontsize=12)

# 2c — label distribution per source (stacked)
pivot = df.groupby(['source', 'label']).size().unstack(fill_value=0)
pivot.plot(kind='bar', ax=axes[2], color=[colors.get(c, '#90A4AE') for c in pivot.columns],
           edgecolor='white', rot=30)
axes[2].set_title('Label distribution per source', fontweight='bold')
axes[2].set_xlabel('')
axes[2].set_ylabel('Count')
axes[2].legend(title='Label')

plt.tight_layout()
plt.savefig('../data/raw/plot_distributions.png', bbox_inches='tight')
plt.show()

## 3. Text-length analysis

In [ ]:
df['char_len'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

print('=== Text length statistics (chars) ===')
print(df.groupby('source')['char_len'].describe().round(1).to_string())
print('\n=== Word count statistics ===')
print(df.groupby('source')['word_count'].describe().round(1).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for src in df['source'].unique():
    sub = df[df['source'] == src]
    p99_char = sub['char_len'].quantile(0.99)
    p99_word = sub['word_count'].quantile(0.99)
    sub['char_len'].clip(upper=p99_char).plot.hist(
        bins=40, alpha=0.55, ax=axes[0], label=src, density=True)
    sub['word_count'].clip(upper=p99_word).plot.hist(
        bins=40, alpha=0.55, ax=axes[1], label=src, density=True)

axes[0].set_title('Character length distribution (99th pct clip)', fontweight='bold')
axes[0].set_xlabel('Characters')
axes[0].legend()
axes[1].set_title('Word count distribution (99th pct clip)', fontweight='bold')
axes[1].set_xlabel('Words')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/raw/plot_text_length.png', bbox_inches='tight')
plt.show()

## 4. Text-length vs sentiment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lbl_palette = {'positive': '#4CAF50', 'negative': '#F44336'}

sns.boxplot(data=df, x='label', y='char_len', hue='label',
            palette=lbl_palette, ax=axes[0], legend=False,
            flierprops={'markerfacecolor': '0.5', 'markersize': 3})
axes[0].set_title('Character length by sentiment', fontweight='bold')
axes[0].set_yscale('log')
axes[0].set_ylabel('Characters (log scale)')

sns.boxplot(data=df, x='label', y='word_count', hue='label',
            palette=lbl_palette, ax=axes[1], legend=False,
            flierprops={'markerfacecolor': '0.5', 'markersize': 3})
axes[1].set_title('Word count by sentiment', fontweight='bold')
axes[1].set_yscale('log')
axes[1].set_ylabel('Words (log scale)')

plt.tight_layout()
plt.savefig('../data/raw/plot_length_vs_sentiment.png', bbox_inches='tight')
plt.show()

## 5. Top-20 words — positive vs negative

In [ ]:
STOPWORDS = {
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'is','it','this','that','was','be','as','i','he','she','they','we',
    'you','my','his','her','not','are','by','from','so','if','do','all',
    'have','has','had','will','would','could','should','just','also','there',
    'been','its','their','about','into','up','out','no','what','when','which',
    'one','more','very','than','even','can','get','got','like','some','then',
    'who','film','book','movie','story','s','t','m','re'
}

def top_words(texts, n=20):
    words = re.findall(r'[a-z]+', ' '.join(texts).lower())
    words = [w for w in words if w not in STOPWORDS and len(w) > 2]
    return Counter(words).most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, lbl, color in zip(axes, ['positive', 'negative'], ['#4CAF50', '#F44336']):
    subset = df[df['label'] == lbl]['text']
    if subset.empty:
        continue
    tw = top_words(subset)
    words20, counts20 = zip(*tw)
    bars = ax.barh(list(reversed(words20)), list(reversed(counts20)), color=color, alpha=0.8)
    ax.set_title(f'Top-20 words — {lbl.upper()}', fontweight='bold', color=color)
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('../data/raw/plot_top20_by_sentiment.png', bbox_inches='tight')
plt.show()

## 6. Top-20 words per source

In [ ]:
sources = df['source'].unique()
n = len(sources)
fig, axes = plt.subplots(1, n, figsize=(7 * n, 6))
if n == 1:
    axes = [axes]

palette = sns.color_palette('tab10', n)

for ax, src, color in zip(axes, sources, palette):
    tw = top_words(df[df['source'] == src]['text'])
    if not tw:
        continue
    ww, cc = zip(*tw)
    ax.barh(list(reversed(ww)), list(reversed(cc)), color=color, alpha=0.85)
    ax.set_title(f'Top-20 words\n{src}', fontweight='bold')
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('../data/raw/plot_top20_per_source.png', bbox_inches='tight')
plt.show()

## 7. Word cloud

In [ ]:
try:
    from wordcloud import WordCloud, STOPWORDS as WC_STOP

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, lbl, cmap in zip(axes, ['positive', 'negative'], ['Greens', 'Reds']):
        text = ' '.join(df[df['label'] == lbl]['text'])
        if not text.strip():
            continue
        wc = WordCloud(
            width=700, height=400, background_color='white',
            stopwords=WC_STOP | STOPWORDS,
            max_words=150, colormap=cmap
        ).generate(text)
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title(f'Word Cloud — {lbl.upper()}', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig('../data/raw/plot_wordcloud.png', bbox_inches='tight')
    plt.show()
except ImportError:
    print('wordcloud not installed — skipping.  pip install wordcloud')

## 8. Summary

In [ ]:
print('=' * 55)
print('  DATASET SUMMARY')
print('=' * 55)
print(f'Total records      : {len(df):,}')
print(f'Unique labels      : {sorted(df["label"].unique())}')
print(f'Class balance      : {dict(df["label"].value_counts())}')
imbalance = df['label'].value_counts()
ratio = imbalance.max() / imbalance.min() if imbalance.min() > 0 else float('inf')
print(f'Imbalance ratio    : {ratio:.2f}x')
print()
print('--- Per-source breakdown ---')
print(df.groupby('source')['label'].value_counts().to_string())
print()
print('--- Average text length (words) ---')
print(df.groupby('source')['word_count'].mean().round(1).to_string())
print('=' * 55)